# NLLB B subsample 4,842M — seed 42 (resume)

Model: NLLB-600M source Kaggle; sample B_sub4842; resume từ ckpt step đã upload.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'no GPU'
p = torch.cuda.get_device_properties(0)
print('gpu', p.name, round(p.total_memory/1e9, 1), 'GB')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], capture_output=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [ ]:
DATA = None
for c in Path('/kaggle/input').rglob('units_B_sub4842.parquet'):
    DATA = c.parent; break
assert DATA, 'không thấy units_B_sub4842'
TOOLS = None
for c in Path('/kaggle/input').rglob('train_nllb.py'):
    TOOLS = c.parent; break
assert TOOLS, 'không thấy tools'
CKPT = None
for c in Path('/kaggle/input').rglob('adapter_config.json'):
    CKPT = c.parent; break
STEP = 0
if CKPT:
    f = CKPT/'STEP.txt'
    if f.exists(): STEP = int(f.read_text().strip() or 0)
MODEL = None
for cfg in Path('/kaggle/input').rglob('config.json'):
    try: d = json.loads(cfg.read_text())
    except Exception: continue
    if d.get('model_type') == 'm2m_100' or 'M2M100' in str(d.get('architectures')):
        MODEL = str(cfg.parent); break
assert MODEL, 'không thấy NLLB model'
print('DATA', DATA, '| TOOLS', TOOLS, '| CKPT', CKPT, 'step', STEP, '| MODEL', MODEL)


In [ ]:
OUT = Path('/kaggle/working/out'); OUT.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(TOOLS/'train_nllb.py'),
       '--units', str(DATA/'units_B_sub4842.parquet'),
       '--sample', str(DATA/'sample_B_sub4842_balanced_seed42.parquet'),
       '--eval', str(DATA/'eval_B.parquet'),
       '--model', MODEL, '--out', str(OUT), '--seed', '42',
       '--steps', '500', '--save-every', '25']
if STEP > 0:
    cmd += ['--init-adapter', str(CKPT), '--start-step', str(STEP)]
print('RUN', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)
src = OUT/'hyp_seed42.parquet'
if src.exists():
    src.rename(OUT/'hyp_nllb_b_sub4842_seed42.parquet')
print('DONE')
